# LangChain: el mismo flujo usando framework

LangChain nos evita escribir a mano varias partes del pipeline.

| Paso | Manual | Con LangChain |
|---|---|---|
| Chunking | Lo programamos nosotros | `RecursiveCharacterTextSplitter` |
| Documento con metadata | Diccionarios | `Document` |
| Embeddings | OpenAI SDK directo | `OpenAIEmbeddings` |
| Vector store | Lista + NumPy | `InMemoryVectorStore` |
| Prompt | String manual | `ChatPromptTemplate` |
| LLM | OpenAI SDK directo | `ChatOpenAI` |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

BASE = Path.cwd()
if not (BASE / 'data' / 'documentos.py').exists(): BASE = BASE.parent
if not (BASE / 'data' / 'documentos.py').exists(): BASE = Path('ai_engineer/ejercicios_embeddings_simple').resolve()
load_dotenv(BASE.parent / '.env'); sys.path.append(str(BASE / 'data'))

from documentos import DOCUMENTOS, PREGUNTA

## 1. Data inicial

Usamos exactamente los mismos documentos del notebook manual. La diferencia es que ahora LangChain crea objetos y operaciones mas comodas.

In [ ]:
for doc in DOCUMENTOS:
    print(f"--- {doc['source']} ---")
    print(doc['text'].strip(), '\n')

print('Pregunta:', PREGUNTA)

## 2. Chunking con `RecursiveCharacterTextSplitter`

Este splitter intenta cortar texto respetando separadores naturales como saltos de linea y espacios.

Parametros importantes:

| Parametro | Valor | Significado |
|---|---:|---|
| `chunk_size` | 220 | Tamano maximo aproximado del chunk en caracteres |
| `chunk_overlap` | 40 | Caracteres repetidos entre chunks vecinos |

El overlap ayuda a que una idea no se pierda si queda justo en el borde entre dos chunks.

In [ ]:
# LangChain corta el texto solo, respetando chunk_size y chunk_overlap.
# Docs: https://python.langchain.com/docs/how_to/recursive_text_splitter/
# API ref: https://python.langchain.com/api_reference/text_splitters/character/langchain_text_splitters.character.RecursiveCharacterTextSplitter.html
splitter = RecursiveCharacterTextSplitter(chunk_size=220, chunk_overlap=40)

docs = splitter.create_documents(
    texts=[doc['text'] for doc in DOCUMENTOS],
    metadatas=[{'source': doc['source']} for doc in DOCUMENTOS],
)

for i, doc in enumerate(docs, start=1):
    print(f"{i}. ({doc.metadata['source']}) {doc.page_content}")

### Grafico: longitud de chunks

Este grafico muestra como quedo la fragmentacion que hizo LangChain.

In [ ]:
longitudes = [len(doc.page_content.split()) for doc in docs]

plt.figure(figsize=(8, 3))
plt.bar(range(1, len(docs) + 1), longitudes)
plt.title('Palabras por chunk en LangChain')
plt.xlabel('Chunk')
plt.ylabel('Palabras')
plt.show()

## 3. Embeddings + vector store

`OpenAIEmbeddings` crea los vectores.

`InMemoryVectorStore` los guarda en memoria y permite hacer busqueda semantica.

En este ejemplo no usamos una base externa porque para aprender alcanza con memoria local.

In [ ]:
# OpenAIEmbeddings docs: https://python.langchain.com/docs/integrations/text_embedding/openai/
# InMemoryVectorStore docs: https://python.langchain.com/docs/integrations/vectorstores/in_memory/
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)

print('Documentos indexados:', len(docs))
print('Vector store:', type(vectorstore).__name__)

## 4. Busqueda Top-K

`similarity_search_with_score` hace dos cosas:

1. Convierte la pregunta en embedding.
2. Busca los chunks mas parecidos en el vector store.

`k=3` significa: devolver los 3 mejores resultados.

In [ ]:
# Le pedimos al vector store los 3 chunks mas parecidos a la pregunta.
# Docs: https://python.langchain.com/docs/concepts/vectorstores/#similarity-search
top_3 = vectorstore.similarity_search_with_score(PREGUNTA, k=3)

for doc, score in top_3:
    print(f"[{score:.4f}] ({doc.metadata['source']}) {doc.page_content}")

### Grafico: scores recuperados

Si el retrieval funciona, los mejores scores deberian venir del documento de red.

In [ ]:
plt.figure(figsize=(6, 3))
plt.bar([doc.metadata['source'] for doc, _ in top_3], [score for _, score in top_3])
plt.title('Scores Top-3 en LangChain')
plt.ylabel('Score')
plt.xticks(rotation=20)
plt.show()

## 5. RAG con `ChatPromptTemplate` y `ChatOpenAI`

Ahora construimos un contexto con los chunks recuperados.

El prompt le dice al modelo que responda usando solo ese contexto. Esa instruccion ayuda a reducir respuestas inventadas.

In [ ]:
# Armamos el contexto, el prompt, y le pedimos la respuesta al modelo.
# ChatPromptTemplate docs: https://python.langchain.com/docs/concepts/prompt_templates/
contexto = '\n\n'.join(
    f"[Fuente {i}] {doc.metadata['source']}: {doc.page_content}"
    for i, (doc, score) in enumerate(top_3, start=1)
)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Responde solo usando el contexto.'),
    ('human', 'Pregunta: {pregunta}\n\nContexto:\n{contexto}'),
])

# ChatOpenAI docs: https://python.langchain.com/docs/integrations/chat/openai/
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
respuesta = llm.invoke(prompt.invoke({'pregunta': PREGUNTA, 'contexto': contexto}))

print('Respuesta RAG:', respuesta.content)

## Cierre

LangChain no cambia la idea central. Solo empaqueta pasos que antes hicimos manualmente:

- splitter para chunks;
- embeddings;
- vector store;
- busqueda Top-K;
- prompt + LLM.